# Tokenization and Stopword Removal on IMDb Movie Reviews

## 1. Project Objective

This project focuses on understanding and implementing basic NLP text preprocessing techniques using the IMDb movie review dataset.

The main objectives are to:

- Understand sentence-level and word-level tokenization.
- Remove unnecessary text noise such as HTML tags and extra whitespace.
- Normalize text through lowercasing.
- Understand and apply English stopword removal.
- Preserve sentiment-important negation words such as **not**.
- Compare raw text with tokenized and cleaned text.
- Measure the effect of preprocessing on token count and vocabulary size.
- Build a modular and reusable NLP preprocessing pipeline.
- Perform quality checks to verify that important text information is not lost.

## 2. Dataset

The IMDb dataset contains 50,000 movie reviews with two main columns:

- **review**  Movie review text.
- **sentiment**  Positive or negative sentiment label.

For development and preprocessing analysis, a balanced subset of 1,000 reviews was selected:

- 500 positive reviews
- 500 negative reviews

## 3. Preprocessing Workflow

    The implemented workflow is:

    Raw Review  
       ↓  
    Noise Removal  
       ↓  
    Lowercasing  
       ↓  
    Sentence Tokenization  
       ↓  
    Word Tokenization  
       ↓  
    Sentiment-Aware Stopword Removal  
       ↓  
    Clean Tokens  
       ↓  
    Before vs. After Analysis  
       ↓  
    Vocabulary Analysis  
       ↓  
    Quality Assurance

## 4. Key Results

The preprocessing pipeline produced the following results on the 1,000-review development subset:

- Raw tokens: 274,114
- Clean tokens: 153,045
- Token reduction: 44.17%
- Raw vocabulary: 20,103
- Clean vocabulary: 19,206
- Vocabulary reduction: 4.46%

The final quality checks confirmed that all 1,000 reviews were successfully processed and important negation information such as **not** was preserved.

##Step 1  Environment Initialization
Objective

Set up the NLP environment and download the NLTK resources required for

1. Sentence tokenization
2. Word tokenization
3. Stopword removal
3. Later normalization/lemmatization if required

### 1.1 Install Required Libraries

In [1]:
# Install required NLP libraries
!pip install -q nltk spacy pandas

### Why These Libraries?

We are using these libraries to make the text preprocessing process easier and more organized.

1.  **NLTK (nltk)**  Used for tokenization, stopword removal, and basic NLP preprocessing.
2.  **spaCy (spacy)**  Used for advanced NLP processing and text analysis.
3.  **Pandas (pandas)**  Used to load, explore, and manipulate the IMDB dataset.
4.  **Regular Expressions (re)**  Used to find and remove unwanted characters and patterns from the text.

**Note:** The re module does not need to be installed because it is already included in Python's standard library.

### 1.2 Import Libraries

In [2]:
import re
import pandas as pd
import nltk

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

### 1.3 Download NLTK Resources

In [3]:

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

### What Each Resource Does

We download these NLTK resources because they provide the data needed for different NLP preprocessing tasks.

1. **punkt**  Used for sentence and word tokenization.
2. **punkt_tab**  Provides tokenizer data required by newer NLTK versions.
3. **stopwords**  Provides a standard list of English stopwords for removal.
4.  **wordnet**  Used later if we apply lemmatization to reduce words to their base forms.

### 1.4 Verify Installation

In [4]:
print("NLTK version:", nltk.__version__)
print("Pandas version:", pd.__version__)

print("\nEnglish stopwords:", len(stopwords.words("english")))

sample = "This is a simple NLP sentence."

print("\nSentence tokens:")
print(sent_tokenize(sample))

print("\nWord tokens:")
print(word_tokenize(sample))

NLTK version: 3.9.1
Pandas version: 2.2.3

English stopwords: 198

Sentence tokens:
['This is a simple NLP sentence.']

Word tokens:
['This', 'is', 'a', 'simple', 'NLP', 'sentence', '.']


## Step 2  IMDb Dataset Loading

### Objective

In this step, we will load the IMDb dataset and take a quick look at its structure before applying any text preprocessing.

To keep development and testing fast, we will work with a **1,000-review subset** of the dataset. This allows us to experiment and iterate quickly while keeping the same preprocessing workflow that can later be applied to the full dataset.

### Step 2.1  Upload/Locate Dataset

In [8]:
import pandas as pd

dataset_path = "/content/IMDB Dataset.csv"

df = pd.read_csv(dataset_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Dataset shape: (50000, 2)

Columns:
['review', 'sentiment']


In [9]:

# Baseline dataset inspection
print("First 5 records:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

First 5 records:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Data types:
review       object
sentiment    object
dtype: object

Missing values:
review       0
sentiment    0
dtype: int64

Duplicate rows:
418

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


### Step 2.2  Create Development Subset

To make development and testing faster, we will create a smaller subset of **1,000 reviews** from the IMDb dataset.

We will keep both sentiment classes equally represented:

- **500 positive reviews**
- **500 negative reviews**

This gives us a balanced development subset while we test our tokenization and stopword removal workflow.

In [10]:
# Create a balanced development subset

positive_reviews = df[df["sentiment"] == "positive"].sample(
    n=500,
    random_state=42
)

negative_reviews = df[df["sentiment"] == "negative"].sample(
    n=500,
    random_state=42
)

df_sample = pd.concat(
    [positive_reviews, negative_reviews],
    ignore_index=True
)

# Shuffle the combined dataset
df_sample = df_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Development subset shape:", df_sample.shape)

print("\nSentiment distribution:")
print(df_sample["sentiment"].value_counts())

Development subset shape: (1000, 2)

Sentiment distribution:
sentiment
negative    500
positive    500
Name: count, dtype: int64


### Step 2.3  Raw Text Inspection

Before applying any cleaning or preprocessing rules, we will first inspect the **actual raw IMDb reviews**.

This helps us understand what the text really looks like, including punctuation, HTML tags, special characters, and other patterns.

We will base our preprocessing steps on these observations rather than making assumptions about the dataset.

In [11]:
# Inspect 5 raw IMDb reviews

for i in range(5):
    print(f"\n--- Review {i + 1} ---")
    print("Sentiment:", df_sample.loc[i, "sentiment"])
    print("Review:")
    print(df_sample.loc[i, "review"])


--- Review 1 ---
Sentiment: negative
Review:
For Daniel Auteuil, `Queen Margot' was much better. For Nastassja Kinski, `Paris, Texas' was much better. The biggest disappointments were from Chris Menges (`CrissCross' and `A World Apart' cannot even be compared with this one), and Goran Bregovic for use of a version of the same musical theme from `Queen Margot' for this movie (Attention to the end of the film). If this was an American pop movie, I would not feel surprised at all; but for a European film with more independent actors and director, a similar common approach about child abuse with no original insight is very simple-minded and disappointing. There are those bad guys who kidnap and sell the underage people. There are those poor children who hate people selling them and wait to be saved by someone. And finally, there is that big hero who kills all the bad guys and saves these poor children from bad guys. Every character is shown in simple black and white terms: the good versus

### Important NLP Observation

After inspecting the raw IMDb reviews, we can identify some important text patterns that will affect our preprocessing.

- Reviews **2 and 3** contain HTML tags such as:
  - `<br /><br />`
- This confirms that **HTML tags should be removed before tokenization**.

We also found sentiment-critical expressions such as:

- `not very original`
- `isn't inspired`
- `don't expect`
- `not disturbing`

These examples show that **negation words are important for sentiment analysis**.

Therefore, we should not blindly remove every NLTK stopword. We need to preserve important negation terms such as **`not`**, **`no`**, **`never`**, and common contracted forms such as **`isn't`** and **`don't`** where appropriate.

## Step 3  Text Cleaning: Noise Removal

In this step, we will create our first text preprocessing function.

For now, we will focus only on two basic cleaning operations:

1. **Remove HTML tags**   such as `<br />` found in the IMDb reviews.
2. **Normalize whitespace**  remove unnecessary spaces, tabs, and line breaks.

We will **not lowercase or tokenize the text yet**.

Keeping each preprocessing stage separate makes the workflow easier to understand, test, and validate.

### Step 3.1  Create Noise Removal Function

In [12]:
def remove_noise(text):
    """
    Remove HTML tags and normalize whitespace
    from a raw IMDb review.
    """

    # Remove HTML tags such as <br /> and <br>
    text = re.sub(r"<[^>]*>", " ", text)

    # Normalize multiple whitespace characters into one space
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

In [13]:
raw_review = df_sample.loc[1, "review"]

cleaned_review = remove_noise(raw_review)

print("RAW REVIEW:\n")
print(raw_review)

print("\n" + "=" * 80 + "\n")

print("CLEANED REVIEW:\n")
print(cleaned_review)

RAW REVIEW:

Spoilers abound. You have been warned.<br /><br />I was thoroughly disappointed, this being my first STREET FIGHTER movie I have seen (I dare not go near the 1994 joke yet). Very little grabs your attention in STREET FIGHTER ZERO (ALPHA) as opposed to most japanimation. The fights are hilariously done over board (Shun versus Zangief was a laugher) and the dramatization is far too moody especially toward the end when Ryu has to control everything in his fight against his brother.<br /><br />The main street fighter, Ryu, has been weakening to a far darker version inside of himself. Frustrations in controlling this darkness are further complicated by the sudden arrival of a younger brother! A shady street fighting tournament is held with more than just fighting on the promoter 's mind.<br /><br />What is with the artist 's drawing of feet? Any anime drawn above the stomach is impressive. The story 's soft nature makes the STREET FIGHTER genre far too intelligent, and places f

### Step 3.2  Lowercasing

Now we will add **lowercasing** to normalize the text.

### Why?

The same word or phrase can appear with different capitalization, for example:

- STREET FIGHTER
- Street Fighter
- street fighter

For basic NLP processing, we generally want these to be treated as the same word:

- street fighter

Lowercasing reduces unnecessary differences caused only by capitalization and makes the text easier to process consistently.

In [14]:
def normalize_text(text):
    """
    Convert text to lowercase and normalize whitespace.
    """

    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text

In [15]:
normalized_review = normalize_text(cleaned_review)

print("BEFORE LOWERCASE:\n")
print(cleaned_review[:1000])

print("\n" + "=" * 80 + "\n")

print("AFTER LOWERCASE:\n")
print(normalized_review[:1000])

BEFORE LOWERCASE:

Spoilers abound. You have been warned. I was thoroughly disappointed, this being my first STREET FIGHTER movie I have seen (I dare not go near the 1994 joke yet). Very little grabs your attention in STREET FIGHTER ZERO (ALPHA) as opposed to most japanimation. The fights are hilariously done over board (Shun versus Zangief was a laugher) and the dramatization is far too moody especially toward the end when Ryu has to control everything in his fight against his brother. The main street fighter, Ryu, has been weakening to a far darker version inside of himself. Frustrations in controlling this darkness are further complicated by the sudden arrival of a younger brother! A shady street fighting tournament is held with more than just fighting on the promoter 's mind. What is with the artist 's drawing of feet? Any anime drawn above the stomach is impressive. The story 's soft nature makes the STREET FIGHTER genre far too intelligent, and places far more emphasis on a chara

### Step 4  Tokenization

Now we reach an important step in NLP **tokenization**.

Tokenization means breaking continuous text into smaller units called **tokens**. These tokens can be complete sentences or individual words.

In this step, we will learn two types of tokenization:

1. **Sentence-level tokenization**  Splits the review into individual sentences.
2. **Word-level tokenization**  Splits the text into individual words and tokens.

We will use the **cleaned and lowercased review** from the previous steps:

```text
normalized_review

### Step 4.1  Sentence-Level Tokenization

Sentence tokenization divides a review into individual sentences.

In [16]:
# Sentence-level tokenization

sentences = sent_tokenize(normalized_review)

print("Number of sentences:", len(sentences))

print("\nFirst 5 sentences:\n")

for i, sentence in enumerate(sentences[:5], start=1):
    print(f"{i}. {sentence}")

Number of sentences: 16

First 5 sentences:

1. spoilers abound.
2. you have been warned.
3. i was thoroughly disappointed, this being my first street fighter movie i have seen (i dare not go near the 1994 joke yet).
4. very little grabs your attention in street fighter zero (alpha) as opposed to most japanimation.
5. the fights are hilariously done over board (shun versus zangief was a laugher) and the dramatization is far too moody especially toward the end when ryu has to control everything in his fight against his brother.


### Why Sentence Tokenization Matters

Sentence tokenization helps us preserve the **sentence-level structure** of a review.

Instead of treating the entire review as one long string, we can work with individual sentences

```text
Sentence 1 → sentiment/context
Sentence 2 → sentiment/context
Sentence 3 → sentiment/context

### Step 4.2  Word-Level Tokenization

Now we'll tokenize the same review into individual words/tokens.

In [17]:
# Word-level tokenization

word_tokens = word_tokenize(normalized_review)

print("Number of tokens:", len(word_tokens))

print("\nFirst 50 tokens:\n")
print(word_tokens[:50])

Number of tokens: 244

First 50 tokens:

['spoilers', 'abound', '.', 'you', 'have', 'been', 'warned', '.', 'i', 'was', 'thoroughly', 'disappointed', ',', 'this', 'being', 'my', 'first', 'street', 'fighter', 'movie', 'i', 'have', 'seen', '(', 'i', 'dare', 'not', 'go', 'near', 'the', '1994', 'joke', 'yet', ')', '.', 'very', 'little', 'grabs', 'your', 'attention', 'in', 'street', 'fighter', 'zero', '(', 'alpha', ')', 'as', 'opposed', 'to']


### Important Observation

After word-level tokenization, punctuation marks can also appear as separate tokens, for example:

```text
'.'
','
'!'
'?'
'('
')'

### Step 4.3  Compare Sentence vs Word Tokenization

In [18]:
print("Sentence-level tokenization:")
print(sentences[:3])

print("\n" + "=" * 80)

print("\nWord-level tokenization:")
print(word_tokens[:30])

Sentence-level tokenization:
['spoilers abound.', 'you have been warned.', 'i was thoroughly disappointed, this being my first street fighter movie i have seen (i dare not go near the 1994 joke yet).']


Word-level tokenization:
['spoilers', 'abound', '.', 'you', 'have', 'been', 'warned', '.', 'i', 'was', 'thoroughly', 'disappointed', ',', 'this', 'being', 'my', 'first', 'street', 'fighter', 'movie', 'i', 'have', 'seen', '(', 'i', 'dare', 'not', 'go', 'near', 'the']


## Step 5  Stopword Removal

Now we'll remove common English stopwords from our word tokens.

### Step 5.1  Inspect NLTK Stopwords

First, let's see what NLTK considers English stopwords.

In [19]:
# Load English stopwords

english_stopwords = set(stopwords.words("english"))

print("Number of stopwords:", len(english_stopwords))

print("\nFirst 50 stopwords:")
print(sorted(english_stopwords)[:50])

Number of stopwords: 198

First 50 stopwords:
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn']


### Step 5.2  Check Important Sentiment Words

Before removing anything, let's specifically inspect words that can change sentiment.

In [25]:
# Check sentiment-important words

important_words = [
    "not",
    "no",
    "nor",
    "never",
    "isn't",
    "wasn't",
    "don't",
    "didn't",
    "can't",
    "couldn't",
    "won't"
]

for word in important_words:
    print(f"{word:10} -> {'STOPWORD' if word in english_stopwords else 'KEEP'}")

not        -> STOPWORD
no         -> STOPWORD
nor        -> STOPWORD
never      -> KEEP
isn't      -> STOPWORD
wasn't     -> STOPWORD
don't      -> STOPWORD
didn't     -> STOPWORD
can't      -> KEEP
couldn't   -> STOPWORD
won't      -> STOPWORD


### Step 5.3  Create a Sentiment-Aware Stopword List

We'll therefore create a custom list that removes standard stopwords except important negation words.

In [21]:
# Preserve negation words because they are important for sentiment analysis

negation_words = {
    "not",
    "no",
    "nor",
    "never",
    "isn't",
    "wasn't",
    "don't",
    "didn't",
    "doesn't",
    "can't",
    "couldn't",
    "won't",
    "wouldn't",
    "shouldn't"
}

sentiment_stopwords = english_stopwords - negation_words

print("Original stopwords:", len(english_stopwords))
print("Sentiment-aware stopwords:", len(sentiment_stopwords))

print("\nPreserved negation words:")
print(sorted(negation_words))

Original stopwords: 198
Sentiment-aware stopwords: 186

Preserved negation words:
["can't", "couldn't", "didn't", "doesn't", "don't", "isn't", 'never', 'no', 'nor', 'not', "shouldn't", "wasn't", "won't", "wouldn't"]


### Step 5.4  Remove Stopwords from Tokens

In [22]:
def remove_stopwords(tokens, stopword_set):
    """
    Remove stopwords while preserving
    sentiment-important words.
    """

    cleaned_tokens = [
        token
        for token in tokens
        if token not in stopword_set
    ]

    return cleaned_tokens

In [23]:
clean_tokens = remove_stopwords(
    word_tokens,
    sentiment_stopwords
)

print("Before stopword removal:")
print(word_tokens[:50])

print("\nAfter stopword removal:")
print(clean_tokens[:50])

Before stopword removal:
['spoilers', 'abound', '.', 'you', 'have', 'been', 'warned', '.', 'i', 'was', 'thoroughly', 'disappointed', ',', 'this', 'being', 'my', 'first', 'street', 'fighter', 'movie', 'i', 'have', 'seen', '(', 'i', 'dare', 'not', 'go', 'near', 'the', '1994', 'joke', 'yet', ')', '.', 'very', 'little', 'grabs', 'your', 'attention', 'in', 'street', 'fighter', 'zero', '(', 'alpha', ')', 'as', 'opposed', 'to']

After stopword removal:
['spoilers', 'abound', '.', 'warned', '.', 'thoroughly', 'disappointed', ',', 'first', 'street', 'fighter', 'movie', 'seen', '(', 'dare', 'not', 'go', 'near', '1994', 'joke', 'yet', ')', '.', 'little', 'grabs', 'attention', 'street', 'fighter', 'zero', '(', 'alpha', ')', 'opposed', 'japanimation', '.', 'fights', 'hilariously', 'done', 'board', '(', 'shun', 'versus', 'zangief', 'laugher', ')', 'dramatization', 'far', 'moody', 'especially', 'toward']


### Step 5.5  Negation Preservation QA

In [26]:
test_texts = [
    "This movie is good.",
    "This movie is not good.",
    "This movie was amazing.",
    "This movie was not amazing.",
    "I don't like this movie.",
    "I really like this movie."
]

for text in test_texts:
    tokens = word_tokenize(text.lower())
    cleaned = remove_stopwords(tokens, sentiment_stopwords)

    print(f"\nOriginal: {text}")
    print(f"Tokens:   {tokens}")
    print(f"Cleaned:  {cleaned}")


Original: This movie is good.
Tokens:   ['this', 'movie', 'is', 'good', '.']
Cleaned:  ['movie', 'good', '.']

Original: This movie is not good.
Tokens:   ['this', 'movie', 'is', 'not', 'good', '.']
Cleaned:  ['movie', 'not', 'good', '.']

Original: This movie was amazing.
Tokens:   ['this', 'movie', 'was', 'amazing', '.']
Cleaned:  ['movie', 'amazing', '.']

Original: This movie was not amazing.
Tokens:   ['this', 'movie', 'was', 'not', 'amazing', '.']
Cleaned:  ['movie', 'not', 'amazing', '.']

Original: I don't like this movie.
Tokens:   ['i', 'do', "n't", 'like', 'this', 'movie', '.']
Cleaned:  ["n't", 'like', 'movie', '.']

Original: I really like this movie.
Tokens:   ['i', 'really', 'like', 'this', 'movie', '.']
Cleaned:  ['really', 'like', 'movie', '.']


## Step 6  Modular Preprocessing Pipeline

Now we will combine the preprocessing steps we have already tested into **one reusable function**.

The goal is to create a clean and consistent pipeline that can be applied to every IMDb review.

The complete flow will be:

```text
Raw Text
   ↓
Noise Removal
   ↓
Lowercasing
   ↓
Word Tokenization
   ↓
Sentiment-Aware Stopword Removal
   ↓
Clean Tokens

### Step 6.1  Create preprocess_text()

In [27]:
def preprocess_text(text):
    """
    Complete preprocessing pipeline for IMDb reviews.

    Steps:
    1. Remove HTML and extra whitespace
    2. Convert text to lowercase
    3. Tokenize into words
    4. Remove sentiment-aware stopwords
    """

    # Step 1: Remove noise
    text = remove_noise(text)

    # Step 2: Lowercase text
    text = normalize_text(text)

    # Step 3: Word tokenization
    tokens = word_tokenize(text)

    # Step 4: Remove stopwords
    cleaned_tokens = remove_stopwords(
        tokens,
        sentiment_stopwords
    )

    return cleaned_tokens

In [28]:
sample_review = df_sample.loc[0, "review"]

processed_tokens = preprocess_text(sample_review)

print("Original review:")
print(sample_review)

print("\nProcessed tokens:")
print(processed_tokens[:50])

Original review:
For Daniel Auteuil, `Queen Margot' was much better. For Nastassja Kinski, `Paris, Texas' was much better. The biggest disappointments were from Chris Menges (`CrissCross' and `A World Apart' cannot even be compared with this one), and Goran Bregovic for use of a version of the same musical theme from `Queen Margot' for this movie (Attention to the end of the film). If this was an American pop movie, I would not feel surprised at all; but for a European film with more independent actors and director, a similar common approach about child abuse with no original insight is very simple-minded and disappointing. There are those bad guys who kidnap and sell the underage people. There are those poor children who hate people selling them and wait to be saved by someone. And finally, there is that big hero who kills all the bad guys and saves these poor children from bad guys. Every character is shown in simple black and white terms: the good versus the evil. Plus, from the ver

### Step 6.2  Add Sentence-Level Tokenization

We already tested **sent_tokenize()** in the previous step.

Now we will add sentence-level tokenization to our reusable preprocessing function. This will allow us to keep both levels of information:

1. **Sentence-level tokens**  The review split into individual sentences.
2. **Word-level cleaned tokens**  The review split into words after cleaning and stopword removal.

Keeping both representations gives us more flexibility for later NLP analysis.

In [29]:
def tokenize_sentences(text):
    """
    Split normalized text into sentences.
    """
    text = remove_noise(text)
    text = normalize_text(text)

    return sent_tokenize(text)


def preprocess_text_with_sentences(text):
    """
    Return both sentence-level and cleaned word-level tokens.
    """

    sentences = tokenize_sentences(text)
    word_tokens = preprocess_text(text)

    return sentences, word_tokens

In [30]:
sentences, cleaned_tokens = preprocess_text_with_sentences(
    df_sample.loc[0, "review"]
)

print("Number of sentences:", len(sentences))

print("\nFirst 3 sentences:")
for i, sentence in enumerate(sentences[:3], start=1):
    print(f"{i}. {sentence}")

print("\nFirst 30 cleaned word tokens:")
print(cleaned_tokens[:30])

Number of sentences: 12

First 3 sentences:
1. for daniel auteuil, `queen margot' was much better.
2. for nastassja kinski, `paris, texas' was much better.
3. the biggest disappointments were from chris menges (`crisscross' and `a world apart' cannot even be compared with this one), and goran bregovic for use of a version of the same musical theme from `queen margot' for this movie (attention to the end of the film).

First 30 cleaned word tokens:
['daniel', 'auteuil', ',', '`', 'queen', 'margot', "'", 'much', 'better', '.', 'nastassja', 'kinski', ',', '`', 'paris', ',', 'texas', "'", 'much', 'better', '.', 'biggest', 'disappointments', 'chris', 'menges', '(', '`', 'crisscross', "'", '`']


## Step 7  Apply the Pipeline to the Dataset

Now we will move from testing individual reviews to applying our preprocessing pipeline to the complete **1,000-review development dataset**.

For each review, the pipeline will generate two new columns:

- **sentences**  Contains the review split into individual sentences.
- **clean_tokens**  Contains the cleaned word-level tokens after stopword removal.

This gives us a structured dataset that we can use for the next stages of NLP processing and analysis.

### Step 7.1  Process All 1,000 Reviews

In [31]:
df_sample["sentences"] = df_sample["review"].apply(
    tokenize_sentences
)

df_sample["clean_tokens"] = df_sample["review"].apply(
    preprocess_text
)

print("Dataset shape:", df_sample.shape)
print("\nNew columns:")
print(df_sample.columns.tolist())

Dataset shape: (1000, 4)

New columns:
['review', 'sentiment', 'sentences', 'clean_tokens']


In [32]:
print("Original review:")
print(df_sample.loc[0, "review"])

print("\nSentence tokens:")
print(df_sample.loc[0, "sentences"])

print("\nClean word tokens:")
print(df_sample.loc[0, "clean_tokens"][:50])

Original review:
For Daniel Auteuil, `Queen Margot' was much better. For Nastassja Kinski, `Paris, Texas' was much better. The biggest disappointments were from Chris Menges (`CrissCross' and `A World Apart' cannot even be compared with this one), and Goran Bregovic for use of a version of the same musical theme from `Queen Margot' for this movie (Attention to the end of the film). If this was an American pop movie, I would not feel surprised at all; but for a European film with more independent actors and director, a similar common approach about child abuse with no original insight is very simple-minded and disappointing. There are those bad guys who kidnap and sell the underage people. There are those poor children who hate people selling them and wait to be saved by someone. And finally, there is that big hero who kills all the bad guys and saves these poor children from bad guys. Every character is shown in simple black and white terms: the good versus the evil. Plus, from the ver

### Step 7.2  Create a Clean Text Version

In [33]:
df_sample["clean_text"] = df_sample["clean_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

print("Original text:")
print(df_sample.loc[0, "review"])

print("\nClean text:")
print(df_sample.loc[0, "clean_text"])

Original text:
For Daniel Auteuil, `Queen Margot' was much better. For Nastassja Kinski, `Paris, Texas' was much better. The biggest disappointments were from Chris Menges (`CrissCross' and `A World Apart' cannot even be compared with this one), and Goran Bregovic for use of a version of the same musical theme from `Queen Margot' for this movie (Attention to the end of the film). If this was an American pop movie, I would not feel surprised at all; but for a European film with more independent actors and director, a similar common approach about child abuse with no original insight is very simple-minded and disappointing. There are those bad guys who kidnap and sell the underage people. There are those poor children who hate people selling them and wait to be saved by someone. And finally, there is that big hero who kills all the bad guys and saves these poor children from bad guys. Every character is shown in simple black and white terms: the good versus the evil. Plus, from the very 

## Step 8  Before vs. After Comparison

Now we need to demonstrate the actual effect of preprocessing across multiple reviews.

We'll compare

1. Original review
2. Cleaned text
3. Original word-token count
4. Cleaned token count
5. Number of tokens removed

### Step 8.1  Create Comparison Table

In [34]:
comparison_df = df_sample.head(5)[
    ["review", "clean_tokens"]
].copy()

comparison_df["original_token_count"] = comparison_df["review"].apply(
    lambda text: len(word_tokenize(text.lower()))
)

comparison_df["clean_token_count"] = comparison_df["clean_tokens"].apply(
    len
)

comparison_df["tokens_removed"] = (
    comparison_df["original_token_count"]
    - comparison_df["clean_token_count"]
)

comparison_df["clean_text"] = comparison_df["clean_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

display(
    comparison_df[
        [
            "review",
            "clean_text",
            "original_token_count",
            "clean_token_count",
            "tokens_removed"
        ]
    ]
)

,review,clean_text,original_token_count,clean_token_count,tokens_removed
0,"For Daniel Auteuil, `Queen Margot' was much be...","daniel auteuil , ` queen margot ' much better ...",266,159,107
1,Spoilers abound. You have been warned.<br /><b...,spoilers abound . warned . thoroughly disappoi...,265,150,115
2,"Where do I start? The plot of the movie, which...","start ? plot movie , love two high school stud...",450,224,226
3,There was a genie played by Shaq His name was ...,"genie played shaq name kazaam , whack rhymes c...",159,100,59
4,Its a very good comedy movie.Ijust liked it.I ...,good comedy movie.ijust liked it.i n't know lo...,371,232,139


##Step 9  Vocabulary Analysis

Now we measure how preprocessing affects the vocabulary.

#### Concept

A token is an individual occurrence:

    movie movie good movie

has 4 tokens.

A vocabulary contains unique tokens:

    movie, good

has 2 unique tokens.

We will calculate:

1. Raw vocabulary size
2. Clean vocabulary size
3. Number of unique tokens removed
4. Vocabulary reduction percentage

### Step 9.1  Calculate Vocabulary Sizes

In [35]:
# Raw word tokens from all reviews
raw_tokens = []

for text in df_sample["review"]:
    raw_tokens.extend(
        word_tokenize(text.lower())
    )

# Clean tokens from the preprocessing pipeline
clean_tokens = []

for tokens in df_sample["clean_tokens"]:
    clean_tokens.extend(tokens)

# Unique vocabularies
raw_vocabulary = set(raw_tokens)
clean_vocabulary = set(clean_tokens)

# Calculate statistics
raw_vocab_size = len(raw_vocabulary)
clean_vocab_size = len(clean_vocabulary)
vocab_reduction = raw_vocab_size - clean_vocab_size
vocab_reduction_percentage = (
    vocab_reduction / raw_vocab_size
) * 100

print("Raw vocabulary size:", raw_vocab_size)
print("Clean vocabulary size:", clean_vocab_size)
print("Unique tokens removed:", vocab_reduction)
print(
    f"Vocabulary reduction: {vocab_reduction_percentage:.2f}%"
)

Raw vocabulary size: 20103
Clean vocabulary size: 19206
Unique tokens removed: 897
Vocabulary reduction: 4.46%


### Step 9.2 Overall Token Reduction

In [36]:
total_raw_tokens = len(raw_tokens)
total_clean_tokens = len(clean_tokens)

total_tokens_removed = (
    total_raw_tokens - total_clean_tokens
)

token_reduction_percentage = (
    total_tokens_removed / total_raw_tokens
) * 100

print("Total raw tokens:", total_raw_tokens)
print("Total clean tokens:", total_clean_tokens)
print("Total tokens removed:", total_tokens_removed)
print(
    f"Token reduction: {token_reduction_percentage:.2f}%"
)

Total raw tokens: 274114
Total clean tokens: 153045
Total tokens removed: 121069
Token reduction: 44.17%


## Step 10  Quality Assurance

Before finishing, we should verify that preprocessing did not accidentally remove important information.

We'll check:

1. Empty reviews after preprocessing
2. Negation preservation
3. Number of reviews processed
4. Sentiment labels remain unchanged

### Step 10.1  Pipeline QA

In [37]:
print("Total reviews:", len(df_sample))

print(
    "Empty cleaned token lists:",
    df_sample["clean_tokens"].apply(len).eq(0).sum()
)

print(
    "Missing clean text:",
    df_sample["clean_text"].isna().sum()
)

print(
    "Sentiment distribution after preprocessing:"
)
print(df_sample["sentiment"].value_counts())

# Check negation preservation
negation_test = [
    "This movie is not good.",
    "This movie is not amazing."
]

print("\nNegation QA:")

for text in negation_test:
    tokens = preprocess_text(text)
    print(f"Original: {text}")
    print(f"Processed: {tokens}")

Total reviews: 1000
Empty cleaned token lists: 0
Missing clean text: 0
Sentiment distribution after preprocessing:
sentiment
negative    500
positive    500
Name: count, dtype: int64

Negation QA:
Original: This movie is not good.
Processed: ['movie', 'not', 'good', '.']
Original: This movie is not amazing.
Processed: ['movie', 'not', 'amazing', '.']


## 11. Key Findings

- The IMDb dataset contains 50,000 reviews, while a balanced subset of 1,000 reviews was used for development and analysis.
- Sentence tokenization successfully divided reviews into individual sentences.
- Word tokenization converted reviews into individual word-level tokens.
- Sentiment-aware stopword removal reduced the total number of tokens while preserving important negation words such as `not`.
- The preprocessing pipeline reduced total tokens from 274,114 to 153,045, resulting in a 44.17% token reduction.
- Vocabulary size decreased from 20,103 to 19,206 unique tokens, representing a 4.46% vocabulary reduction.
- Quality checks found no empty cleaned token lists and no missing cleaned text values.
- The positive and negative sentiment distribution remained balanced at 500 reviews per class.
- The QA tests confirmed that phrases such as **not good** and **not amazing** retained their negation information.

## 13. Conclusion

This project demonstrated a practical NLP preprocessing workflow for IMDb movie reviews. The workflow covered noise removal, lowercasing, sentence tokenization, word tokenization, and sentiment-aware stopword removal.

The preprocessing pipeline reduced the total number of token occurrences by 44.17%, while the unique vocabulary decreased by 4.46%. This shows that removing frequent stopwords can significantly reduce the amount of text processed without removing a large portion of the unique vocabulary.

The sentiment-aware approach also preserved important negation information such as **not**, which is important for sentiment-related NLP tasks. Final quality checks confirmed that all 1,000 reviews were processed successfully, with no empty cleaned outputs or missing cleaned text values.

Overall, the project established a reusable and modular preprocessing pipeline that can be used as a foundation for further NLP tasks such as text classification, sentiment analysis, and feature extraction.